In [2]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs

# Configuración de conexión a MinIO en el VPS
MINIO_ENDPOINT = "http://169.58.96.176:9000"
MINIO_KEY = "admin"      # Reemplaza con tu Access Key real
MINIO_SECRET = "adminGrupo3"  # Reemplaza con tu Secret Key real
BUCKET_NAME = "programacion-ciencia-datos-g3"

storage_opts = {
    "key": MINIO_KEY,
    "secret": MINIO_SECRET,
    "client_kwargs": {"endpoint_url": MINIO_ENDPOINT}
}

fs = s3fs.S3FileSystem(
    key=MINIO_KEY, 
    secret=MINIO_SECRET, 
    client_kwargs={"endpoint_url": MINIO_ENDPOINT}
)

BATCH_SIZE = 100000

# ==============================================================================
# 1. TRANSFORMACIÓN DE TRANSACCIONES (CON FECHAS REALES DESDE 2026-01-01)
# ==============================================================================
print("⚙️ 1. Transformando train_transaction_clean.parquet con proyecciones de fecha...")

ruta_in_trans = f"{BUCKET_NAME}/silver/train_transaction_clean.parquet"
ruta_out_trans = f"{BUCKET_NAME}/silver/train_transaction_transformed.parquet"

cols_seleccionadas_trans = [
    'TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
    'ProductCD', 'card1', 'card2', 'card4', 'card6',
    'addr1', 'dist1', 'P_emaildomain', 'C1', 'D1'
]

pf_trans = pq.ParquetFile(ruta_in_trans, filesystem=fs)
writer_trans_tf = None
resumen_parcial_grupos = []
total_tx_procesadas = 0

# Fecha base oficial del proyecto (1 de enero de 2026)
FECHA_BASE = pd.Timestamp('2026-01-01 00:00:00')

with fs.open(ruta_out_trans, 'wb') as f_out:
    for i, batch in enumerate(pf_trans.iter_batches(batch_size=BATCH_SIZE, columns=cols_seleccionadas_trans)):
        chunk = batch.to_pandas()
        
        # A. Conversión temporal: Mapeo de segundos relativos a fecha y hora de calendario real
        chunk['fecha_hora'] = FECHA_BASE + pd.to_timedelta(chunk['TransactionDT'], unit='s')
        chunk['fecha_dia'] = chunk['fecha_hora'].dt.date.astype(str)
        chunk['hora_dia'] = chunk['fecha_hora'].dt.hour.astype('int8')
        chunk['dia_semana'] = chunk['fecha_hora'].dt.day_name()
        chunk['mes_operacion'] = chunk['fecha_hora'].dt.month_name()
        
        # B. Franja horaria comercial
        chunk['franja_horaria'] = pd.cut(
            chunk['hora_dia'],
            bins=[-1, 5, 11, 17, 23],
            labels=['Madrugada (00-05h)', 'Mañana (06-11h)', 'Tarde (12-17h)', 'Noche (18-23h)']
        ).astype(str)
        
        # C. Conversión a Soles Peruanos (PEN - Caso Izipay) y segmentación de ticket
        chunk['monto_pen'] = (chunk['TransactionAmt'] * 3.75).round(2).astype('float32')
        chunk['rango_ticket'] = pd.cut(
            chunk['TransactionAmt'],
            bins=[0, 15, 50, 200, np.inf],
            labels=['1_Micro (<$15)', '2_Bajo ($15-$50)', '3_Medio ($50-$200)', '4_Alto (>$200)']
        ).astype(str)
        
        # D. Pre-agrupación por bloque para análisis de fraude por día de la semana y tarjeta
        grupo_chunk = chunk[chunk['card4'] != 'desconocido'].groupby(['card4', 'dia_semana'], as_index=False).agg(
            conteo_tx=('TransactionID', 'count'),
            suma_fraudes=('isFraud', 'sum'),
            suma_monto_usd=('TransactionAmt', 'sum')
        )
        resumen_parcial_grupos.append(grupo_chunk)
        
        # E. Escritura incremental del bloque transformado
        tabla_pa = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer_trans_tf is None:
            writer_trans_tf = pq.ParquetWriter(f_out, tabla_pa.schema, compression='snappy')
        writer_trans_tf.write_table(tabla_pa)
        
        total_tx_procesadas += len(chunk)
        print(f"   • Bloque Transacciones #{i+1} procesado ({len(chunk):,} filas)")

    if writer_trans_tf:
        writer_trans_tf.close()

print(f"✅ train_transaction_transformed.parquet guardado con éxito ({total_tx_procesadas:,} filas).\n")


# ==============================================================================
# 2. TRANSFORMACIÓN DE IDENTIDADES POR CHUNKS (CAPA SILVER)
# ==============================================================================
print("⚙️ 2. Transformando train_identity_clean.parquet por bloques...")

ruta_in_id = f"{BUCKET_NAME}/silver/train_identity_clean.parquet"
ruta_out_id = f"{BUCKET_NAME}/silver/train_identity_transformed.parquet"

cols_seleccionadas_id = ['TransactionID', 'id_01', 'id_02', 'id_12', 'id_15', 'id_30', 'id_31', 'DeviceType', 'DeviceInfo']

pf_id = pq.ParquetFile(ruta_in_id, filesystem=fs)
writer_id_tf = None

with fs.open(ruta_out_id, 'wb') as f_out_id:
    for i, batch_id in enumerate(pf_id.iter_batches(batch_size=BATCH_SIZE, columns=cols_seleccionadas_id)):
        chunk_id = batch_id.to_pandas()
        
        # Simplificación de familia de Sistema Operativo
        s_os = chunk_id['id_30'].astype(str).str.lower()
        condiciones = [
            s_os.str.contains('windows'),
            s_os.str.contains('ios'),
            s_os.str.contains('android'),
            s_os.str.contains('mac'),
            s_os.str.contains('linux')
        ]
        opciones = ['Windows', 'iOS', 'Android', 'MacOS', 'Linux']
        chunk_id['familia_os'] = np.select(condiciones, opciones, default='Otro/Desconocido')
        
        # Escritura incremental
        tabla_pa_id = pa.Table.from_pandas(chunk_id, preserve_index=False)
        if writer_id_tf is None:
            writer_id_tf = pq.ParquetWriter(f_out_id, tabla_pa_id.schema, compression='snappy')
        writer_id_tf.write_table(tabla_pa_id)
        print(f"   • Bloque Identidad #{i+1} procesado ({len(chunk_id):,} filas)")

    if writer_id_tf:
        writer_id_tf.close()

print("✅ train_identity_transformed.parquet guardado en Capa Silver!\n")


# ==============================================================================
# 3. RESULTADOS DE LA AGRUPACIÓN GLOBAL POR DÍA DE LA SEMANA
# ==============================================================================
print("📊 === TABLA ANALÍTICA: TASA DE FRAUDE POR DÍA DE LA SEMANA Y MARCA DE TARJETA ===")
df_agrupado_dias = pd.concat(resumen_parcial_grupos, ignore_index=True)
df_agrupado_dias = df_agrupado_dias.groupby(['card4', 'dia_semana'], as_index=False).sum()
df_agrupado_dias['tasa_fraude_pct'] = (df_agrupado_dias['suma_fraudes'] / df_agrupado_dias['conteo_tx'] * 100).round(2)
display(df_agrupado_dias.sort_values(by=['card4', 'conteo_tx'], ascending=[True, False]).head(14))

⚙️ 1. Transformando train_transaction_clean.parquet con proyecciones de fecha...
   • Bloque Transacciones #1 procesado (100,000 filas)
   • Bloque Transacciones #2 procesado (100,000 filas)
   • Bloque Transacciones #3 procesado (100,000 filas)
   • Bloque Transacciones #4 procesado (100,000 filas)
   • Bloque Transacciones #5 procesado (100,000 filas)
   • Bloque Transacciones #6 procesado (90,540 filas)
✅ train_transaction_transformed.parquet guardado con éxito (590,540 filas).

⚙️ 2. Transformando train_identity_clean.parquet por bloques...
   • Bloque Identidad #1 procesado (100,000 filas)
   • Bloque Identidad #2 procesado (44,233 filas)
✅ train_identity_transformed.parquet guardado en Capa Silver!

📊 === TABLA ANALÍTICA: TASA DE FRAUDE POR DÍA DE LA SEMANA Y MARCA DE TARJETA ===


,card4,dia_semana,conteo_tx,suma_fraudes,suma_monto_usd,tasa_fraude_pct
0,american express,Friday,1377,49,253452.000000,3.56
6,american express,Wednesday,1309,38,245777.156250,2.90
4,american express,Thursday,1291,26,241649.187500,2.01
1,american express,Monday,1251,34,197269.250000,2.72
5,american express,Tuesday,1198,23,216026.250000,1.92
2,american express,Saturday,952,47,148499.750000,4.94
3,american express,Sunday,950,22,138546.250000,2.32
8,discover,Monday,1072,90,310958.375000,8.40
7,discover,Friday,1032,71,297690.906250,6.88
12,discover,Tuesday,1028,73,280817.250000,7.10


In [3]:
import pandas as pd
import s3fs

MINIO_ENDPOINT = "http://169.58.96.176:9000"
MINIO_KEY = "admin"
MINIO_SECRET = "adminGrupo3"
BUCKET_NAME = "programacion-ciencia-datos-g3"

storage_opts = {
    "key": MINIO_KEY,
    "secret": MINIO_SECRET,
    "client_kwargs": {"endpoint_url": MINIO_ENDPOINT}
}

print("🔍 Consultando transacciones de marzo de 2026...")

# 1. Leer el archivo Parquet transformado de transacciones
df_trans = pd.read_parquet(
    f"s3://{BUCKET_NAME}/silver/train_transaction_transformed.parquet", 
    storage_options=storage_opts
)

# 2. Asegurar que la columna 'fecha_hora' sea de tipo datetime
df_trans['fecha_hora'] = pd.to_datetime(df_trans['fecha_hora'])

# 3. Filtrar exclusivamente las transacciones de MARZO DE 2026 (Mes 03)
df_marzo = df_trans[(df_trans['fecha_hora'].dt.year == 2026) & (df_trans['fecha_hora'].dt.month == 3)]

print(f"✨ Total de transacciones encontradas en marzo de 2026: {len(df_marzo):,}")

# 4. Agrupar para ver el resumen de fraudes vs. legítimas en ese mes
resumen_fraude_marzo = df_marzo.groupby('isFraud').agg(
    total_transacciones=('TransactionID', 'count'),
    monto_total_usd=('TransactionAmt', 'sum'),
    monto_total_pen=('monto_pen', 'sum')
).reset_index()

# Mapear la etiqueta de fraude para mayor claridad visual
resumen_fraude_marzo['estado'] = resumen_fraude_marzo['isFraud'].map({0: 'Legítima (No Fraude)', 1: 'Fraude Confirmado'})

print("\n📊 Resumen Operativo - Marzo 2026:")
display(resumen_fraude_marzo[['estado', 'total_transacciones', 'monto_total_usd', 'monto_total_pen']])

print("\n🔎 Muestra de transacciones individuales en marzo de 2026:")
display(df_marzo[['TransactionID', 'fecha_hora', 'card4', 'TransactionAmt', 'monto_pen', 'isFraud']].head(10))

🔍 Consultando transacciones de marzo de 2026...
✨ Total de transacciones encontradas en marzo de 2026: 94,170

📊 Resumen Operativo - Marzo 2026:


,estado,total_transacciones,monto_total_usd,monto_total_pen
0,Legítima (No Fraude),90302,12482520.00,46809380.0
1,Fraude Confirmado,3868,534419.75,2004073.5



🔎 Muestra de transacciones individuales en marzo de 2026:


,TransactionID,fecha_hora,card4,TransactionAmt,monto_pen,isFraud
218404,3205404,2026-03-01 00:00:07,visa,49.000000,183.750000,0
218405,3205405,2026-03-01 00:00:29,mastercard,46.644001,174.919998,0
218406,3205406,2026-03-01 00:01:19,visa,57.000000,213.750000,0
218407,3205407,2026-03-01 00:01:20,visa,113.949997,427.309998,0
218408,3205408,2026-03-01 00:01:26,visa,59.000000,221.250000,1
218409,3205409,2026-03-01 00:01:28,visa,49.000000,183.750000,0
218410,3205410,2026-03-01 00:01:52,mastercard,20.000000,75.000000,0
218411,3205411,2026-03-01 00:02:13,visa,39.000000,146.250000,0
218412,3205412,2026-03-01 00:03:00,mastercard,85.192001,319.470001,1
218413,3205413,2026-03-01 00:03:04,mastercard,46.644001,174.919998,0
